#Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import trim,col

In [0]:
rename_map = {'sls_ord_num':'order_number',
              'sls_prd_key':'product_key',
              'sls_cust_id':'sales_customer_id',
              'sls_order_dt':'order_date',
              'sls_ship_dt':'ship_date',
              'sls_due_dt':'due_date',
              'sls_sales':'sales_amount',
              'sls_quantity':'quantity',
              'sls_price':'price'}

#Reading Data From Bronze Layer

In [0]:
df = spark.table('workspace.bronze.crm_sales_details')

#Data Cleaning

##Trimming String columns

In [0]:
for column in df.dtypes:
    if column[1] == 'string':
        df = df.withColumn(column[0],trim(col(column[0])))

##Type Casting

In [0]:
df = df.withColumn('sls_order_dt', F.try_to_date(df['sls_order_dt'], 'yyyyMMdd'))
df = df.withColumn('sls_ship_dt', F.try_to_date(df['sls_ship_dt'], 'yyyyMMdd'))
df = df.withColumn('sls_due_dt', F.try_to_date(df['sls_due_dt'], 'yyyyMMdd'))

##Handling Null Values

In [0]:
df = df.withColumn('sls_sales',F.coalesce(df['sls_sales'],F.lit(0)))
df = df.withColumn('sls_quantity',F.coalesce(df['sls_quantity'],F.lit(0)))
df = df.withColumn('sls_price',F.coalesce(df['sls_price'],F.lit(0)))

##Handling Invalid Values 

In [0]:
df = df.withColumn('sls_price',F.abs(df['sls_price']))
df =(
    df.withColumn(
        'sls_sales',
        F.when(df['sls_sales'] != df['sls_quantity'] * df['sls_price'], df['sls_quantity'] * df['sls_price'])
        .when(df['sls_sales'] <= 0, df['sls_quantity'] * df['sls_price'])
        .otherwise(df['sls_sales'])
    )
)

##Standardization

###Column Naming Standardization

In [0]:
for old_name,new_name in rename_map.items():
    df = df.withColumnRenamed(old_name,new_name)

#Loading Data Into Silver Layer

In [0]:
df.write.mode('overwrite').format('delta').saveAsTable('silver.crm_sales_details')